Gruppenarbeit:  


- Artjom Poprjaduhha (1929317)  


- Ben Schell (1934117)  


- Dirk Bender (673327)  

## Modellierungsseminar Sommer 2026

## Cycle Planning for workforce scheduling

### load packages

In [ ]:
# install Gurobi package in case not done yet:
#%pip install gurobipy 
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
from dataclasses import dataclass
import datetime as dt
from pathlib import Path # for easier and robust folder and file handling across OS (Path can be used by Pandas directly)
import src.Shift as Shift # tailor-made data type for shift definitions
import src.functions as abd # self-made functions by Arty, Ben and Dirk... ;-) => call them by starting with "abd."


### setting parameters and variables

#### global variables

In [ ]:
# determine folder structure for inputs and outputs
PROJECT_ROOT = Path.cwd() # root folder of the code
FOLDER_INPUT =  PROJECT_ROOT / "input" # data input
FOLDER_LOGS = PROJECT_ROOT / "logs" # folder for log files, eg exports of data sets for more transparency
FOLDER_OUTPUT = PROJECT_ROOT / "output/"
FOLDER_AND_FILE_LOG =  PROJECT_ROOT / "logs" / "cyclePlanning_logs.txt"


In [ ]:
abd.writeToLogs("cycle planning process started", FOLDER_AND_FILE_LOG, deleteHistory=True) # very first log entry deletes old log entries

In [ ]:
# load parameters from CSV
params = abd.readParameters(FOLDER_INPUT / "parameters.csv")
abd.writeToLogs("parameters loaded", FOLDER_AND_FILE_LOG)

#### boundaries for feasible solutions and objective prioritization

In [ ]:
# global static variables from imported parameters.csv file

# more relevant for more complex models going forward
MAX_CYCLE_WEEKS = int(params["max_cycle_length"])   # max number of cycle weeks
MIN_CYCLE_WEEKS = int(params["min_cycle_length"])   # min number of cycle weeks
MAX_NB_CYCLES = int(params["max_nb_cycles"])        # number of cycles
MIN_NB_CYCLES = int(params["min_nb_cycles"])        # number of cycles


# variables for fairness targets, average weekly work hours, presence time, rest time, free days, etc (used as weighted for objective function)
TARGET_WEEKLY_HOURS     = float(params["avg_weekly_work_hours"])    # target avg weekly work hours
AVG_REFERENCE_WEEKS     = int(params["avg_reference_weeks"])        # nb of weeks in window for average (currently unused, reserved)
MAX_WEEKLY_HOURS        = float(params["max_weekly_work_hours"])    # hard cap on work hours per cycle week
MAX_FREE_DAYS_PER_WEEK  = int(params["max_free_days_per_week"])     # max fully free days per active week
MAX_CONSEC_FREE_DAYS    = int(params["max_consec_free_days"])       # rolling cap, max free days IN A ROW (optimizer tended to fill up cycles with free days to meet other objectives)
MIN_REST                = int(params["min_rest"])                   # min rest time between shifts in minutes
MAX_CONSEC_DAYS         = int(params["max_conseq_working_days"])    # max consecutive working days
CLONE_RATIO_RESERVE     = float(params["reserve_clone_ratio"])      # quote for cloning shifts to create reserve stand-bys (based on TuMuPl-file)


# weights for various selectable objectives (user input)
W_Min_NB_WORKERS  = int(params["obj_w_workers"])                                    # weight: minimize active cycle weeks (across all cycles)
W_EQUAL_AVG_WEEKLY_PRESENCE_CYCLE = int(params["obj_w_eq_avg_presence"])            # weight: "Pesch0" equal average weekly presence per cycle (from Pesch email)
W_TARGET_WEEKLY_HOURS = int(params["obj_w_minDev_target_weekly_hours"])             # weight: "Pesch1" deviation from target weekly hours
W_EQUAL_SHIFT_CAT_PROPORTION = int(params["obj_w_eq_class_prop"])                   # weight: "Pesch2" equal shift-categorie proportion across cycles
W_MIN_CHANGEofTYPE_CYCLE = int(params["obj_w_min_total_type_changes"])              # weight: "Pesch3" type changes, total over the Turnusauswahl (letter criterion 3)
W_EQUAL_WEEKLY_PRESENCE_CYCLE = int(params["obj_w_eq_weekly_presence"])             # weight: "Pesch4" equal weekly presence across the weeks inside each cycle
W_MIN_CHANGEofTYPE_CYCLEPLAN = int(params["obj_w_min_shift_type_chg_cycleplan"])    # weight: "Pesch5" type changes inside each Turnusgruppe, min-max over cycles (letter criterion 5)
W_EVEN_WEEKEND_DUTIES = int(params["obj_w_eq_weekend_duty"])                        # weight: "Pesch6" (even weekend-duty load across cycles)
W_EVEN_NIGHT_SHIFT_DISTRIBUTION = int(params["obj_w_eq_night_duty"])                # weight: "Pesch7" (even night-shift load across cycles)
W_MIN_CHANGEofCLASSES_d2d = int(params["obj_w_minChangeOfClasses"])                 # weight: "Pesch8" objective to minimize change of shift classes 


In [ ]:
# basic inputs and parameters
cycles = range(1, MAX_NB_CYCLES+1)          # empty shell - not all cycles might be activated during optimization
cycleWeeks  = range(1, MAX_CYCLE_WEEKS+1)   # empty shell - not all cycle weeks might be activated during optimization
Weekdays = range(1,8)                       # results in 1,...,7 => let 1 be Monday and 7 be Sunday (in line with static variable DICT_WEEKDAYS)

In [ ]:
# dictionaries for weekday conversions (string to int and vice versa), 
# used for easier handling of weekdays in the model and for output of results;
# giving the user flexibility to use different formats for weekdays in the input data (eg "Mon" or "Mo" or "1" for Monday)
DICT_WEEKDAYS = {'Mon':1,'Tue':2,'Wed':3,'Thu':4,'Fri':5,'Sat':6,'Sun':7,
                 'Monday':1,'Tuesday':2,'Wednesday':3,'Thursday':4,'Friday':5,'Saturday':6,'Sunday':7,
                 'Mo':1,'Tu':2,'We':3,'Th':4,'Fr':5,'Sa':6,'Su':7,
                 '1':1,'2':2,'3':3,'4':4,'5':5,'6':6,'7':7
                 }
DICT_WEEKDAYS_RETURN = {1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat", 7: "Sun"}

### modelling

#### loading shift objects

In [ ]:

# FROM FILE: read input data for shift definitions
data_shiftSet = abd.readShiftSet(filename=FOLDER_INPUT / "input_ShiftDataSet_InstanzPesch.csv", clone_ratio=CLONE_RATIO_RESERVE)
shift_objects = abd.build_shift_objects(data_shiftSet)

# distinguish "work shift" (WorkShifts) from "all shifts" (Shifts): 
#   WorkShifts are all shifts imported from the shift set file incl. reserve clones
#   the only non-work shift is the hard-coded freeDay below
WorkShifts = [s.shift_id for s in shift_objects] #object oriented solution
abd.writeToLogs(f"WorkShifts are defined as {WorkShifts}", FOLDER_AND_FILE_LOG)
abd.writeDataToLogs(WorkShifts, FOLDER_LOGS / "log_workShift_object.csv")

# additional hard-coded shift for free days
freeDayShift = Shift.Shift("[freeDay_:-)_]", 
                           "dummy shift for free days", 
                           ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], 
                           dt.time(*map(int,"07:00".split(':'))), # arbitrary time
                           dt.time(*map(int,"07:00".split(':'))), # arbitrary time
                           0, # required staff
                           1, # shift class (1: very attractive, 10: least attractive)
                           0, # is_work_shift = False
                           0, # required qualifications (not implemented in this model version)
                           None 
                           )
shift_objects.append(freeDayShift)

#log
abd.writeDataToLogs(shift_objects, FOLDER_LOGS / "log_shift_object.csv")

Shifts = [s.shift_id for s in shift_objects]
abd.writeToLogs(f"    Shifts are defined as {Shifts}", FOLDER_AND_FILE_LOG)

DUMMY_SHIFTS = {"[freeDay_:-)_]"}
NO_HOURS_SHIFTS = {freeDayShift.shift_id}

##### shift info (for output)

In [ ]:
# preparation for output
shift_info = {
    s.shift_id: {
        "start": s.start.strftime("%H:%M"),
        "end": s.end.strftime("%H:%M"),
        "workingTime": s.shift_work_time_assignment
    }
    for s in shift_objects
}

##### incompatible pairs of shifts  
_(prohibited combinations based on minimum rest time)_

In [ ]:
# PROHIBITED SHIFT PAIRS

# pre-compute all shift pairs that violate MIN_REST if scheduled on consecutive days
# excludes the freeDay
# returns a list of (sh1_id, sh2_id) tuples that must not appear on consecutive days within a cycle.

incompatible_pairs = [
    (sh1.shift_id, sh2.shift_id)
    for sh1 in shift_objects if sh1.shift_id not in DUMMY_SHIFTS
    for sh2 in shift_objects if sh2.shift_id not in DUMMY_SHIFTS
    if Shift.rest_minutes_between(sh1, sh2) < MIN_REST
]
abd.writeDataToLogs(incompatible_pairs, FOLDER_LOGS / "log_incompatible_shift_pairs.csv")

# identify night shifts by clock time (end <= start = runs past midnight)
# used for even distribution of night shifts across cycles (if W_EVEN_NIGHT_SHIFT_DISTRIBUTION > 0)
# NOTE: this approach is assuming there is no shift longing for more than 24 hrs!
overnight_ids = [s.shift_id for s in shift_objects
                 if s.shift_id not in DUMMY_SHIFTS and s.end <= s.start]
abd.writeDataToLogs(overnight_ids, FOLDER_LOGS / "log_night_shift_object.csv")


##### presence time and work time of shifts

In [ ]:
# pre-compute work hours per shift using shift_duration_hours from Shift.py.

# required presence (including break if applicable); fall back to presence time if not specified ('none')
shift_hours = {
    s.shift_id: Shift.shift_duration_hours(s)
    for s in shift_objects
    if s.shift_id not in NO_HOURS_SHIFTS 
}

# net work time per shift (break excluded); fall back to presence time if not specified ('none')
work_hours = {
    s.shift_id: (float(s.shift_work_time_assignment)
                 if s.shift_work_time_assignment is not None
                 else Shift.shift_duration_hours(s))
    for s in shift_objects
    if s.shift_id not in NO_HOURS_SHIFTS 
}

#### initiate model

In [ ]:
# modelling
# create a new Gurobi object for the cycle planning problem
# increased robustness: if Gurobi license is not found at default places, user is asked to select the license file via a GUI file dialog
import os
import tkinter as tk
from tkinter import filedialog

try:
    modelCycle = gp.Model("cyclePlanning")
except gp.GurobiError as e:
    print(f"standard Gurobi model creation failed: {e}")
    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)

        lic_path = filedialog.askopenfilename(
            title="Select your Gurobi license file (gurobi.lic)",
            filetypes=[("Gurobi license", "*.lic"), ("All files", "*.*")]
        )
        root.destroy()

        if not lic_path:
            raise RuntimeError("No license file selected.")

        os.environ["GRB_LICENSE_FILE"] = lic_path
        print(f"Using license file: {lic_path}")

        # retry after user selection
        modelCycle = gp.Model("cyclePlanning")

    except Exception as inner_e:
        raise RuntimeError(f"Could not initialize Gurobi model with selected license: {inner_e}")
except Exception as e:
    raise RuntimeError(f"Could not initialize Gurobi model: {e}")

#### basic decision variables

$x_{c,w,d,s} \quad \to \quad $ x  
$y_{c,w} \quad \to \quad $ active\_cycle\_week  
$z_{c} \quad \to \quad $ active\_cycle

In [ ]:
# --- decision variables
# x[c, w, d, s] = 1 if in cycle c, cycleWeek w, on weekday d, shift s is assigned
x = modelCycle.addVars(cycles, cycleWeeks, Weekdays, Shifts,
                       vtype=GRB.BINARY, name="x")

# active_cycle[c] = 1 if cycle c is used at all, 0 otherwise
active_cycle = modelCycle.addVars(cycles, vtype=GRB.BINARY, name="active_cycle")

# active_cycle_week[c,w] = 1 if cycle c uses linked cycleWeek w (i.e., at least one shift in that week is active), 0 otherwise
active_cycle_week = modelCycle.addVars(cycles, cycleWeeks, vtype=GRB.BINARY, name="active_cycle_week")

### constraints:

##### supporting functions

supporting functions:  

 - convert a global day index t into tuple (cycleWeek, weekday)  
 - make the schedule a cycle  

In [ ]:
# supporting function: convert a global day index t into tuple (cycleWeek, weekday)
# => allows to treat all days across all cycleWeeks as one continuous timeline.
def decode_global_day(t):
    week = (t - 1) // 7 + 1 # cycleWeek index from global day t (1-based indexing)
    day = (t - 1) % 7 + 1 # weekday index from global day t (1-based indexing)
    return week, day


In [ ]:
# function to support making the schedule a cycle (ring closure)
def last_active_expr(c, w):
    if w < MAX_CYCLE_WEEKS:
        return active_cycle_week[c, w] - active_cycle_week[c, w + 1]
    return active_cycle_week[c, w]

#### basic constraints: relevant for general functionality

In [ ]:
# [c01]
# ensure a cycleWeek can only be active when the according cylce is active
for c in cycles:
    for w in cycleWeeks:
        modelCycle.addConstr(active_cycle_week[c,w] - active_cycle[c] <= 0,
                             name=f"WeekImpliesCycle_c{c}_s{w}")


##### constraint: minimum number of active cycles 

In [ ]:
# [c02]
# minimum number of active cycles requested by the input parameters
modelCycle.addConstr(
    gp.quicksum(active_cycle[c] for c in cycles) - MIN_NB_CYCLES >= 0,
    name="MinActiveCycles"
)

# [c03]
# NOTE: maximum number of active cycles is implicitly limited by the number of cycles defined in the model (MAX_NB_CYCLES)

##### constraint: min and max number of cycleWeeks per cycle

In [ ]:
# [c04] [c05]
# a cycleWeek can only be active if the corresponding cycle is active (already enforced above, redundant here but not harming)
# if a cycle is active, it must contain between MIN_CYCLE_WEEKS and MAX_CYCLE_WEEKS active weeks
for c in cycles:
    modelCycle.addConstr(
        gp.quicksum(active_cycle_week[c, w] for w in cycleWeeks) <= MAX_CYCLE_WEEKS * active_cycle[c],
        name=f"Link_activeWeek_activeCycle_upper_c{c}"
    )
    modelCycle.addConstr(
        gp.quicksum(active_cycle_week[c, w] for w in cycleWeeks) >= MIN_CYCLE_WEEKS * active_cycle[c],
        name=f"Link_activeWeek_activeCycle_lower_c{c}"
    )

# NOTE: the upper bound is redundant as the number of cycleWeeks is already limited by the model definition (MAX_CYCLE_WEEKS)

##### constraint: shift must only be assigned to active cycleWeeks  

In [ ]:
# [c06]
# shifts must only be assigned if the week is active;
# enforce active_week >= any assignment in that week
# further, if a given cycleWeek is active, at least one shift must be assigned in that week (otherwise the week would be considered inactive)
# NOTE: this constraint does NOT enforce any minimum number of shifts per active week!
for c in cycles:
    for w in cycleWeeks:
        modelCycle.addConstr(
            gp.quicksum(x[c, w, d, sh] for d in Weekdays for sh in Shifts) <= len(Weekdays) * len(Shifts) * active_cycle_week[c, w], # "BIG-M approach"
            name=f"Link_x_activeWeek_c{c}_w{w}_upper"
        )

##### constraint: ensure that cycles and cycleWeeks are activated in ascending order  
_(important for output and ring closure)_

In [ ]:
# [c07] [c08]
# enforce that cycles and cycleWeeks are used in order (no gaps)
for c in range(1, MAX_NB_CYCLES): # loop from first to second-last entry
    modelCycle.addConstr(active_cycle[c] - active_cycle[c + 1] >= 0,
                         name=f"CycleOrder_c{c}")

for c in cycles: # loop across all possible cycles
    for w in range(1, MAX_CYCLE_WEEKS): # loop from first to second-last entry
        modelCycle.addConstr(active_cycle_week[c, w] - active_cycle_week[c, w + 1] >= 0,
                             name=f"WeekOrder_c{c}_s{w}")

# note: this is not only for cosmetical reasons, but also very helpful for the ring-closure of some constraints

##### constraint: each shift must be covered as required  

each shift (more precise: WorkShift) has to be covered on each relevant day (as defined in the shift set) exactly once (given the shift is required for that weekday only)  


In [ ]:
# [c09]
# ensure each shift relevant for a given weekday is covered exactly once across all cycles and cycleWeeks
for day in Weekdays: #loop over all week days
    for ws in WorkShifts: # loop over all work shift
        shift = next(s for s in shift_objects if s.shift_id == ws) # calling shift_object s for WorkShift ws (iteration over ws)
        if day in [DICT_WEEKDAYS[w] for w in shift.weekdays]: # info: this also covers shifts that are required / do appear for non-consecutive week days (eg. Mon, Wed - not Tue)
            modelCycle.addConstr(gp.quicksum(x[c, w, day, ws] for c in cycles for w in cycleWeeks) == 1,
                        name=f"Cover_day{day}_{ws}")
        else: # ensures that shifts that are not required for a given weekday are really not assigned in the model 
            modelCycle.addConstr(gp.quicksum(x[c, w, day, ws] for c in cycles for w in cycleWeeks) <= 0,
                        name=f"Cover_day{day}_{ws}")


##### constraint: each active cycle must have assigned EXACTLY one shift per day in each active cycleWeek  

$ \sum_{s \in S}{x_{c,w,d,s}} - y_{c,w} = 0$
  
$ \quad \forall c \in C, \forall w \in W, \forall d \in D $


In [ ]:
# [c10]
# each cycle shall have EXACTLY one shift per day across all cycleWeeks (implying that cycleWeek has assigned exactly one shift per day)
for c in cycles:
    for w in cycleWeeks:
        for day in Weekdays:
            modelCycle.addConstr(
                gp.quicksum(x[c, w, day, sh] for sh in Shifts) - active_cycle_week[c, w] == 0,
                name=f"OneShiftPerDay_c{c}_s{w}_d{day}"
            )

##### constraint: limit number of consecutive working days

In [ ]:
# [c11a]
# constraint: ensure propper rest time as defined in input parameters (MAX_CONSEC_DAYS)
# => there is a maximum number of consecutive working days allowed across all cycleWeeks for each cycle

# total number of days across all cycleWeeks
# used to define the global timeline over which consecutive work days are checked.
TOTAL_DAYS = MAX_CYCLE_WEEKS * 7 # note: this implies that the model assumes a 7-day week and that all cycleWeeks are of equal length (7 days) - would need to be adjusted if cycleWeeks of different lengths are allowed in the future.

# limit the number of consecutive working days across all cycleWeeks
# for each cycle scan through the entire global timeline and check every (MAX_CONSEC_DAYS + 1)-days window
# to ensure that not all days in the window are working days.
for c in cycles:
    # iterate over all possible start positions of a sliding window
    # the window length is MAX_CONSEC_DAYS + 1, so the last valid start is:
    # TOTAL_DAYS - MAX_CONSEC_DAYS
    for t_start in range(1, TOTAL_DAYS - MAX_CONSEC_DAYS + 1):
        # Collect expressions representing work indicators for each day in the window
        moving_time_window = []
        # iterate through each offset inside the window
        # offset = 0 means the first day of the window
        # offset = MAX_CONSEC_DAYS means the last day of the window
        for offset in range(0, MAX_CONSEC_DAYS + 1):
            # Compute the global day index inside the window
            t = t_start + offset
            # convert global day index back to (cycleWeek, weekday)
            w, d = decode_global_day(t)
            # work[c,w,d] = sum of all work shifts assigned on that day
            # If any WorkShift is assigned, this sum becomes 1 (binary model)
            moving_time_window.append(gp.quicksum(x[c, w, d, ws] for ws in WorkShifts))

        # RELATED CONSTRAINT:
        # In any time window of length MAX_CONSEC_DAYS + 1 (=> implying 1 free day as sufficient)
        # the number of working days must be <= MAX_CONSEC_DAYS.
        modelCycle.addConstr(gp.quicksum(moving_time_window) <= MAX_CONSEC_DAYS, name=f"MaxConsecDays_c{c}_t{t_start}")

# [c11b]
    # ring closure: ensure the constraint is working for last active cycleWeek transitioning into first active cycleWeek
    for w in cycleWeeks:
        for k in range(1, MAX_CONSEC_DAYS + 1):
            tail = gp.quicksum(x[c, w, d, sh] for d in range(8 - k, 8) for sh in WorkShifts) # last k days of week w
            head = gp.quicksum(x[c, 1, d, sh] for d in range(1, MAX_CONSEC_DAYS + 2 - k) for sh in WorkShifts) # first (MAX+1-k) days of week 1

            modelCycle.addConstr(tail + head <= MAX_CONSEC_DAYS + (MAX_CONSEC_DAYS + 1) * (1 - last_active_expr(c, w)), name=f"RingMaxConsec_c{c}_w{w}_k{k}")

# note: there is no objective function for this constraint, it is a hard constraint that must be satisfied in any feasible solution.

##### constraint: minimum rest time between shifts

In [ ]:
# [c12a]
# constraint: minimum rest time between consecutive shifts within a snake week.
# if sh1 on day d and sh2 on day d+1 violate MIN_REST, they cannot both be assigned to the same cycleWeek

# implicitly implemented by support of incompatible_pairs (created in shift definition block above)

for c in cycles:
    for w in cycleWeeks:
        for day in Weekdays:
            if day <= 6:  # for Mon to Sat use pairs of day d and d+1
                for (sh1, sh2) in incompatible_pairs:
                    modelCycle.addConstr(
                        x[c, w, day, sh1] + x[c, w, day+1, sh2] <= 1,
                        name=f"MinRest_c{c}_w{w}_d{day}_{sh1}_{sh2}"
                    )
            elif day == 7 and w < MAX_CYCLE_WEEKS:  # Sun of week w -> Mon of NEXT week w+1 (snake runs continuously)
                for (sh1, sh2) in incompatible_pairs:
                    modelCycle.addConstr(
                        x[c, w, day, sh1] + x[c, w+1, 1, sh2] <= 1,
                        name=f"MinRest_c{c}_w{w}_d{day}_{sh1}_{sh2}"
                    )        

# [c12b]
# ring closure
for c in cycles:
    for w in cycleWeeks:
        for (sh1, sh2) in incompatible_pairs:
            modelCycle.addConstr(
                x[c, w, 7, sh1] + x[c, 1, 1, sh2] <= 2 - last_active_expr(c, w),
                name=f"RingMinRest_c{c}_w{w}_{sh1}_{sh2}"
            )


##### constraint: limit weekly working time (excluding breaks)  

In [ ]:
# [c13]
# constraint: total work hours per active cycle week must not exceed MAX_WEEKLY_HOURS, actual number from parameters.csv
# note: this is implemented by using the dictionairy containing work time per shift as defined above

for c in cycles:
    for w in cycleWeeks:
        modelCycle.addConstr(
            gp.quicksum(work_hours[sh] * x[c, w, d, sh] for d in Weekdays for sh in work_hours.keys()) - MAX_WEEKLY_HOURS * active_cycle_week[c, w] <= 0,
            name=f"MaxWeeklyHours_c{c}_w{w}"
        )

##### constraint: balanced cycle lengths    
ensure balanced [almost equal] length of cycles (same number of cycleWeeks plus/minus 1 week)  

_note: this constraint comes on top of min and max cycleWeeks_

In [ ]:
# [c14]
# ensure all used cycles have balanced number of cycleWeeks;
# deviations of at most 1 week across all cycles are allowed

# dictionary: determine number of active weeks per cycle
cycle_length = {c: gp.quicksum(active_cycle_week[c, s] for s in cycleWeeks) for c in cycles}

# pairwise balance constraints
# |W_c1 - W_c2| <= 1  for all cycles c1 != c2 => implemented as 2 linear statements
for c1 in cycles:
    for c2 in cycles:
        if c1 < c2:  # avoiding duplicates and self-pairing
            # W_c1 - W_c2 <= 1
            modelCycle.addConstr(
                cycle_length[c1] - cycle_length[c2] <= 1 + MAX_CYCLE_WEEKS * (1 - active_cycle[c2]), name=f"CycleBalance_upper_c{c1}_c{c2}") ## upper bound: cycle c1 cannot be more than 1 week longer than c2 — relaxed if c2 is inactive
            # W_c2 - W_c1 <= 1
            modelCycle.addConstr(cycle_length[c2] - cycle_length[c1] <= 1 + MAX_CYCLE_WEEKS * (1 - active_cycle[c1]), name=f"CycleBalance_lower_c{c1}_c{c2}") ## lower bound: cycle c2 cannot be more than 1 week longer than c1 — relaxed if c1 is inactive

# note: by multiplying by MAX_CYCLE_WEEKS, this constraint is using quite high numbers for relaxation, just to be on the safe side

##### constraint: limit number of freeDays within one cycleWeek
max fully free days per active week to suppress the solver filling up the schedule with free days to reach other objectives

In [ ]:
# [c15]
# constraint: at most MAX_FREE_DAYS_PER_WEEK completely free days per active week.
# note: this constraint is looking into cycleWeeks, not in a k-days-window!
for c in cycles:
    for w in cycleWeeks:
        modelCycle.addConstr(gp.quicksum(x[c, w, d, freeDayShift.shift_id] for d in Weekdays) - (MAX_FREE_DAYS_PER_WEEK * active_cycle_week[c, w]) <= 0,
            name=f"MaxFreeDays_c{c}_w{w}"
        )

##### constraint: limit number of free days in a row

In [ ]:
# [c16]
# at most MAX_CONSEC_FREE_DAYS free days IN A ROW
FREE_ID = freeDayShift.shift_id
for c in cycles:
    for t_start in range(1, TOTAL_DAYS - MAX_CONSEC_FREE_DAYS + 1):
        window = []
        for offset in range(0, MAX_CONSEC_FREE_DAYS + 1):
            w, day = decode_global_day(t_start + offset)
            window.append(x[c, w, day, FREE_ID])
        modelCycle.addConstr(
            gp.quicksum(window) <= MAX_CONSEC_FREE_DAYS,
            name=f"MaxConsecFree_c{c}_t{t_start}"
        )

# ring closure: 
for c in cycles:
    for w in cycleWeeks:
        for k in range(1, MAX_CONSEC_FREE_DAYS + 1):
            tail = gp.quicksum(x[c, w, d, FREE_ID] for d in range(8 - k, 8))
            head = gp.quicksum(x[c, 1, d, FREE_ID] for d in range(1, MAX_CONSEC_FREE_DAYS + 2 - k))
            modelCycle.addConstr(
                tail + head <= MAX_CONSEC_FREE_DAYS
                             + (MAX_CONSEC_FREE_DAYS + 1) * (1 - last_active_expr(c, w)),
                name=f"RingMaxConsecFree_c{c}_w{w}_k{k}"
            )

### objectives and their linked constraints

#### objective: minimize number of workers (aka min active cycleWeeks)

$$ \min {\sum_{c \in C} {\sum_{w \in W} y_{c,w}}} $$


In [ ]:
# [o01]
obj_term_min_nb_workers = gp.quicksum(active_cycle_week[c, w] for c in cycles for w in cycleWeeks)

#### objective: minimize deviation of net work time from the weekly target

$ h_{c,w}^{work} = \sum_{d \in D} \sum_{s \in s^{work}} h^{work}_{s} \cdot x_{c,w,d,sh} $  
  
$ \quad \forall c \in C, w \in W $

In [ ]:
# [o02]
# ("Pesch1")
# objective: minimize deviation of net work time from the weekly target (AVG_WEEKLY_HOURS) across all cycles and cycleWeeks 
# note: the objective penalises deviation, it does NOT force exactly 40h!
# Measured per week so heavy/light weeks cannot cancel out across the cycle.

# new variable to sum up the total work hours per cycle/cycleWeek across all shifts assigned in that week (used for objective function)
week_hours = {(c, w): gp.quicksum(work_hours[sh] * x[c, w, d, sh] for d in Weekdays for sh in work_hours.keys()) for c in cycles for w in cycleWeeks}

# new slack variables the solver can vary for optimization (used in constraint below)
# => keep the model linear even though the actual objective is to minimize the absolute deviation
dev_pos_time = modelCycle.addVars(cycles, cycleWeeks, lb=0, name="dev_pos")
dev_neg_time = modelCycle.addVars(cycles, cycleWeeks, lb=0, name="dev_neg")

# [c17]
# new constraint to link new solver-variables with the model
for c in cycles:
    for w in cycleWeeks:
        modelCycle.addConstr(
            week_hours[(c, w)] - TARGET_WEEKLY_HOURS * active_cycle_week[c, w] == dev_pos_time[c, w] - dev_neg_time[c, w],
            name=f"Pesch1_dev_c{c}_w{w}"
        )

# term for objective function
obj_term_target_weekly_hours = gp.quicksum(dev_pos_time[c, w] + dev_neg_time[c, w] for c in cycles for w in cycleWeeks)

#### objective: aim at equal proportion of shift categories (by presence time) across cycles

In [ ]:
# [o03]
# ("Pesch2")
# equal proportion of shift categories (by presence time) across cycles.
# category is interpreted as shiftID
# measure presence time (not working time)
shift_cat_of = {s.shift_id: s.shift_id.split("%_%", 1)[0] for s in shift_objects if s.shift_id not in NO_HOURS_SHIFTS}
categories = sorted(set(shift_cat_of.values()))

# dict for presence hours of each category k inside each cycle c
cat_hours = {
    (c, k): gp.quicksum(shift_hours[sh] * x[c, w, d, sh]
                        for w in cycleWeeks for d in Weekdays
                        for sh in shift_hours.keys() if shift_cat_of[sh] == k)
    for c in cycles for k in categories
}

# p_k: share of class k in the dataset's coverage-required presence hours
dataset_cat_hours = {k: 0.0 for k in categories}
for w in shift_objects:
    if w.shift_id in shift_cat_of:
        dataset_cat_hours[shift_cat_of[w.shift_id]] += shift_hours[w.shift_id] * len(w.weekdays)
p = {k: dataset_cat_hours[k] / sum(dataset_cat_hours.values()) for k in categories}

# total presence hours of given categorie within each cycle
total_presence_cat = {c: gp.quicksum(cat_hours[c, k] for k in categories) for c in cycles}

# variables for solver to vary for optimization (used in constraint and objective term below)
# => keep the model linear even though the actual objective is to minimize the absolute deviation
dev_pos_cat = modelCycle.addVars(cycles, categories, lb=0, name="pesch2_pos")
dev_neg_cat = modelCycle.addVars(cycles, categories, lb=0, name="pesch2_neg")

#[c18]
# deviation of class k in cycle c from its fair share of that cycle's hours
for c in cycles:
    for k in categories:
        modelCycle.addConstr(
            cat_hours[c, k] - p[k] * total_presence_cat[c] == dev_pos_cat[c, k] - dev_neg_cat[c, k],
            name=f"Pesch2_share_c{c}_k{k}"
        )

# related term for objective function: sum of deviations across all cycles and categories
obj_term_equal_shift_cat_proportion = (gp.quicksum(dev_pos_cat[c, k] + dev_neg_cat[c, k] for c in cycles for k in categories))

#### objective: minimize day-to-day changes of shift CLASS

In [ ]:
# [o04]
# ("Pesch8")
# objective: minimize change of shift classes from day to day (across all cycles and cycleWeeks)
# target: same shift-class in blocks as far as possible

# new variable for this new objective

# map each shiftID to its shift_class (eg 'dayShift': 2; 'nightShift': 7)
dict_shift_to_class = {s.shift_id: s.shift_class for s in shift_objects}

# unique list - ie set - of shift classes from shift definition (from input file)
set_classes = sorted(set(dict_shift_to_class.values()))

new decision variable: class_assigned[c,w,d,k] = 1 if on cycle c, week w, weekday d the assigned shift belongs to class k:  
  
$ x^{class}_{c,w,d,k} $  
  
$ d^{class}_{c,w,d,k} $

In [ ]:
# additional decision variables to link shift classes to active shifts/shiftWeek/weekDay/cycle:

# new binary variable indicating whether a specific class is assigned on (c,w,d), ie in cycle c, in cycleWeek w, on weekDay d
# (from x point of view: replacing shiftID by its class for this objective)
# class_assigned[c,w,d,k] = 1 if on cycle c, week w, weekday d the assigned shift belongs to class k
class_assigned = modelCycle.addVars(cycles, cycleWeeks, Weekdays, set_classes, vtype=GRB.BINARY, name="class_assigned")

# add supporting variable for absolute differences per class between consecutive days
# (binary as change can happen or not)
# diff[c,w,d,k] >= | class_assigned[c,w,d,k] - class_assigned[c,w,d+1,k] |
class_diff = modelCycle.addVars(cycles, cycleWeeks, Weekdays, set_classes, vtype=GRB.BINARY, name="diff_class")

# link class_assigned to x
# for each shiftClass k, class_assigned equals the sum of x over all shifts that belong to that class
# this enforces that class_assigned is 1 exactly when a shift of that class is chosen on that day
dict_shifts_by_class = {k: [sh for sh, shiftClass in dict_shift_to_class.items() if shiftClass == k and sh in Shifts] for k in set_classes}
# results in sth like this (using example values):
    # 2: ["[00day000week]%_%1", "[00day000week]%_%2", ...],
    # 5: ["[00dayweekend]%_%1", ...],
    # 7: ["[night000week]%_%1", "[night000week]%_%2", ...],
    # 9: ["[nightweekend]%_%1", ...],
    #10: ["[allDay_shift]%_%1"]
    # => focus on WorkShifts - ie excluding free days - would be crucial here if number of freeDays in a row was not limited!

additional constraints to link class_assigned to x, for each cycle, week, day, and class:  
  


In [ ]:
# [c19]
# additional constraints to link class_assigned to x, for each cycle, week, day, and class
for c in cycles:
    for w in cycleWeeks:
        for day in Weekdays:
            for k in set_classes: 
                shifts_in_class_k = dict_shifts_by_class[k] # loads all shift if given class k
                # sum_x_for_class is the expression sum(x[c,w,d,sh] for sh in shifts_in_class_k)
                # Add equality: class_assigned[c,w,d,k] - sum_x_for_class == 0
                if shifts_in_class_k: # run the following code only for non-empty lists of shifts in class k
                    modelCycle.addConstr(
                        class_assigned[c, w, day, k] - gp.quicksum(x[c, w, day, sh] for sh in shifts_in_class_k) == 0,
                        name=f"LinkClass_c{c}_w{w}_d{day}_k{k}"
                    )
                else:
                    # if no shifts for this class (shouldn't happen by definition), force 0 (just a safety net)
                    modelCycle.addConstr(class_assigned[c, w, day, k] == 0,
                                         name=f"LinkClassEmpty_c{c}_w{w}_d{day}_k{k}")
                    
# [c20]
# class_diff constraints for consecutive days: counting class changes (looping over cycle>week>day)
for c in cycles:
    for w in cycleWeeks:                     
        for day in Weekdays:
            # determine next day index (wrap 7 -> 1)
            if day <= 6:
                d_next = day + 1
                w_next = w
            elif w < MAX_CYCLE_WEEKS:  # day == 7: Sunday -> Monday of the NEXT week
                d_next = 1
                w_next = w+1  # use first day of next cycleWeek
            else: # last cycleWeek, loop back to first cycleWeek ("ring closure")
                continue  # Sunday of the very last week: there is no next week here;
                          # the wrap back to week 1 is handled by the RING CLOSURE cell below
            for k in set_classes:
                # pair of two linear constraints to replicate absolute difference
                modelCycle.addConstr(
                    class_diff[c, w, day, k] >= class_assigned[c, w, day, k] - class_assigned[c, w_next, d_next, k] - (1 - active_cycle_week[c, w_next]),
                    name=f"ClassDiffPos_c{c}_w{w}_d{day}_k{k}"
                )
                modelCycle.addConstr(
                    class_diff[c, w, day, k] >= class_assigned[c, w_next, d_next, k] - class_assigned[c, w, day, k] - (1 - active_cycle_week[c, w_next]),
                    name=f"ClassDiffNeg_c{c}_w{w}_d{day}_k{k}"
                )
# Note: diff variables will be 0 when class is same, and 1 when class differs for that k.
# When classes differ (A vs B), two diffs (for A and B) become 1, so sum_k_diff = 2 => to avoid double counting, we multiply by 0.5 in the objective function

class_changes_total = 0.5 * gp.quicksum(class_diff[c, w, d, k] for c in cycles for w in cycleWeeks for d in Weekdays for k in set_classes)

In [ ]:
# [c21]
# ring closure for class_diff: ensure that the last active Sunday of a cycle is compared to the first active Monday of that cycle (week 1)
# (jump from very last Sunday to very first Monday of the same cycle, if both weeks are active)

for c in cycles:
    for w in cycleWeeks:
        for k in set_classes:
            modelCycle.addConstr(
                class_diff[c, w, 7, k] >= class_assigned[c, w, 7, k] - class_assigned[c, 1, 1, k]
                                          - (1 - last_active_expr(c, w)),
                name=f"RingClassDiffPos_c{c}_w{w}_k{k}"
            )
            modelCycle.addConstr(
                class_diff[c, w, 7, k] >= class_assigned[c, 1, 1, k] - class_assigned[c, w, 7, k]
                                          - (1 - last_active_expr(c, w)),
                name=f"RingClassDiffNeg_c{c}_w{w}_k{k}"
            )

#### objective: minimize day-by-day changes of the shift type (ie shiftID)

In [ ]:
# [o05]
# ("Pesch3") 
# objective: aim to keep the SAME shift type (by shiftID) on consecutive days (across all cycles)

# type = base shift name without the artificial %_% copy suffix, e.g. "[00day000week]"
dict_shift_to_type = {sh: sh.split("%_%", 1)[0] for sh in Shifts}
#dict_shift_to_type = {sh: sh.split("%_%", 1)[0] for sh in WorkShifts}
set_types = sorted(set(dict_shift_to_type.values()))
dict_shifts_by_type = {t: [sh for sh, tt in dict_shift_to_type.items() if tt == t] for t in set_types}

# type_assigned[c,w,d,t] = 1 if the shift worked on (c,w,d) is of type t
type_assigned = modelCycle.addVars(cycles, cycleWeeks, Weekdays, set_types, vtype=GRB.BINARY, name="type_assigned")
# type_diff >= |type_assigned(day) - type_assigned(next day)| per type
type_diff = modelCycle.addVars(cycles, cycleWeeks, Weekdays, set_types, vtype=GRB.BINARY, name="type_diff")

# [c22]
# link type_assigned to x
for c in cycles:
    for w in cycleWeeks:
        for day in Weekdays:
            for t in set_types:
                modelCycle.addConstr(
                    type_assigned[c, w, day, t] - gp.quicksum(x[c, w, day, sh] for sh in dict_shifts_by_type[t]) == 0,
                    name=f"LinkType_c{c}_w{w}_d{day}_{t}"
                )

# [c23]
# count type changes between consecutive days; gating and week handling mirror the
for c in cycles:
    for w in cycleWeeks:
        for day in Weekdays:
            if day <= 6:
                d_next, w_next = day + 1, w
            elif w < MAX_CYCLE_WEEKS:
                d_next, w_next = 1, w + 1
            else:
                continue  # wrap of the very last week is handled by the ring part below
            for t in set_types:
                modelCycle.addConstr(
                    type_diff[c, w, day, t] >= type_assigned[c, w, day, t] - type_assigned[c, w_next, d_next, t]
                                             - (1 - active_cycle_week[c, w_next]),
                    name=f"TypeDiffPos_c{c}_w{w}_d{day}_{t}"
                )
                modelCycle.addConstr(
                    type_diff[c, w, day, t] >= type_assigned[c, w_next, d_next, t] - type_assigned[c, w, day, t]
                                             - (1 - active_cycle_week[c, w_next]),
                    name=f"TypeDiffNeg_c{c}_w{w}_d{day}_{t}"
                )

#[c24]
# ring closure
for c in cycles:
    for w in cycleWeeks:
        for t in set_types:
            modelCycle.addConstr(
                type_diff[c, w, 7, t] >= type_assigned[c, w, 7, t] - type_assigned[c, 1, 1, t]
                                         - (1 - last_active_expr(c, w)),
                name=f"RingTypeDiffPos_c{c}_w{w}_{t}"
            )
            modelCycle.addConstr(
                type_diff[c, w, 7, t] >= type_assigned[c, 1, 1, t] - type_assigned[c, w, 7, t]
                                         - (1 - last_active_expr(c, w)),
                name=f"RingTypeDiffNeg_c{c}_w{w}_{t}"
            )

# one real change flips two types (1->0 and 0->1), hence the 0.5 factor (just a constant, though)
type_changes_total = 0.5 * gp.quicksum(type_diff[c, w, d, t] for c in cycles for w in cycleWeeks for d in Weekdays for t in set_types)

#### objective: minimize shift-type changes within each cycle

In [ ]:
# [o06]
# # ("Pesch5")
# objective: minimize type changes within each cycle
# this enriches 'minimization of day-by-day changes' by focusing on the worst-case cycle and therefore aiming at fair distribution

cycle_type_changes = {
    c: 0.5 * gp.quicksum(type_diff[c, w, d, t]
                         for w in cycleWeeks for d in Weekdays for t in set_types)
    for c in cycles
}

max_changes_pan_cycles = modelCycle.addVar(lb=0, name="max_changes_type_per_cycle")

# [c25]
for c in cycles:
    modelCycle.addConstr(max_changes_pan_cycles >= cycle_type_changes[c], name=f"max_changes_type_per_cycle_bound_c{c}")

# note for the objective term: max_changes_pan_cycles is a stand-alone variable that is to be minimized, so no additional formula here


#### objective: equal presence balance within cycles  

In [ ]:
# [o07]
# ("Pesch4")
# objective: equal weekly presence WITHIN each cycle (Turnusgruppe)
# pres_week[c,w] = gross presence hours of week w in cycle c
presence_per_week = {(c, w): gp.quicksum(shift_hours[sh] * x[c, w, d, sh] for d in Weekdays for sh in shift_hours) for c in cycles for w in cycleWeeks}

MAX_WEEK_PRESENCE = 7 * 24   # small gating constant (168h ~ full week) to ensure that inactive weeks are not considered in the objective function

max_presence_cycle = modelCycle.addVars(cycles, lb=0, name="max_presence_cycle")
min_presence_cycle = modelCycle.addVars(cycles, lb=0, name="min_presence_cycle")
for c in cycles:
    # [c26]
    modelCycle.addConstr(min_presence_cycle[c] <= max_presence_cycle[c], name=f"Pesch4_anchor_c{c}")
    for w in cycleWeeks:
    # [c27]
        modelCycle.addConstr(
            max_presence_cycle[c] >= presence_per_week[(c, w)] - MAX_WEEK_PRESENCE * (1 - active_cycle_week[c, w]),
            name=f"presence_upper_c{c}_w{w}"
        )
    # [c28]
        modelCycle.addConstr(
            min_presence_cycle[c] <= presence_per_week[(c, w)] + MAX_WEEK_PRESENCE * (1 - active_cycle_week[c, w]),
            name=f"presence_lower_c{c}_w{w}"
        )

# related objective term
obj_term_equal_weekly_presence_cycle = gp.quicksum(max_presence_cycle[c] - min_presence_cycle[c] for c in cycles)

#### objective: equal presence balance across cycles

In [ ]:
# [o08]
# ("Pesch0")
# objective: equal AVERAGE weekly presence across cycles (Turnusgruppen)
# reusing 'presence_per_week' from above

cycle_pairs = [(c1, c2) for c1 in cycles for c2 in cycles if c1 < c2]

# total presence hours in cycle c
total_presence_cycle = {c: gp.quicksum(presence_per_week[(c, w)] for w in cycleWeeks) for c in cycles}

# number of active weeks cycle c 
active_weeks_cycle = {c: gp.quicksum(active_cycle_week[c, w] for w in cycleWeeks) for c in cycles}

# prep for calculation of average_presence per cycle [c]: total_presence_cycle[c] / active_weeks_cycle[c]
WEEK_COUNT_VALUES = list(range(MIN_CYCLE_WEEKS, MAX_CYCLE_WEEKS + 1))
MAX_PRESENCE_PER_CYCLE = MAX_CYCLE_WEEKS * 7 * max(shift_hours.values())

# selector: exactly one feasible week count is chosen for each active cycle
week_count_sel = modelCycle.addVars(cycles, WEEK_COUNT_VALUES, vtype=GRB.BINARY, name="pesch0_week_count_sel")

# helper vars: z[c, n] = total_presence_cycle[c] * week_count_sel[c, n]
z_presence_by_n = modelCycle.addVars(cycles, WEEK_COUNT_VALUES, lb=0, name="pesch0_z_presence_by_n")

# avg weekly presence per cycle
avg_presence_cycle = modelCycle.addVars(cycles, lb=0, name="pesch0_avg_weekly_presence")

for c in cycles:
    # [c29]
    # choose one week-count value if cycle is active
    modelCycle.addConstr(
        gp.quicksum(week_count_sel[c, n] for n in WEEK_COUNT_VALUES) == active_cycle[c],
        name=f"Pesch0_sel_active_c{c}"
    )

    # [c30]
    # selected week-count must match the number of active weeks
    modelCycle.addConstr(
        gp.quicksum(n * week_count_sel[c, n] for n in WEEK_COUNT_VALUES) == active_weeks_cycle[c],
        name=f"Pesch0_sel_match_weeks_c{c}"
    )

    # total_presence * selector, then determine average
    for n in WEEK_COUNT_VALUES:
    # [c31]
        modelCycle.addConstr(
            z_presence_by_n[c, n] <= MAX_PRESENCE_PER_CYCLE * week_count_sel[c, n],
            name=f"Pesch0_z_ub_sel_c{c}_n{n}"
        )
    # [c32]
        modelCycle.addConstr(
            z_presence_by_n[c, n] <= total_presence_cycle[c],
            name=f"Pesch0_z_ub_tot_c{c}_n{n}"
        )
    # [c33]
        modelCycle.addConstr(
            z_presence_by_n[c, n] >= total_presence_cycle[c] - MAX_PRESENCE_PER_CYCLE * (1 - week_count_sel[c, n]),
            name=f"Pesch0_z_lb_c{c}_n{n}"
        )
    # [c34]
    modelCycle.addConstr(
        avg_presence_cycle[c] == gp.quicksum((1.0 / n) * z_presence_by_n[c, n] for n in WEEK_COUNT_VALUES),
        name=f"Pesch0_avg_def_c{c}"
    )

# pairwise absolute deviations on avg weekly presence
dev_presence_pan_cycles = modelCycle.addVars(cycle_pairs, lb=0, name="dev_presence_pan_cycles")

for (c1, c2) in cycle_pairs: # indicator constraint: considered only if c2 is active
    # [c35]
    modelCycle.addGenConstrIndicator(
        active_cycle[c2], True, avg_presence_cycle[c1] - avg_presence_cycle[c2] <= dev_presence_pan_cycles[c1, c2],
        name=f"dev_avg_presence_upper_c{c1}_{c2}"
    )
    # [c36]
    modelCycle.addGenConstrIndicator(
        active_cycle[c2], True, avg_presence_cycle[c2] - avg_presence_cycle[c1] <= dev_presence_pan_cycles[c1, c2],
        name=f"dev_avg_presence_lower_c{c1}_{c2}"
    )

# related objective term
obj_term_equal_avg_weekly_presence_cycle = gp.quicksum(dev_presence_pan_cycles[c1, c2] for (c1, c2) in cycle_pairs)

#### objective: equal night-shift load across cycles

In [ ]:
# [o09]
# ("Pesch7") 
# objective: equal night presence across cycles (Turnusgruppen)

pair_cycles = range(1, MAX_NB_CYCLES)                                # pairs (c, c+1)
huge_number = MAX_CYCLE_WEEKS * 7 * max(shift_hours.values())        # loose upper bound on cycle hours

# note: night shifts are detected by shift times (start/end) via overnight_ids
NIGHT_IDS = set(overnight_ids)

"""
# For every cycle c, add up the presence hours of all night shifts in it.
# We go over every week, every weekday and every shift, and only keep the shifts
# whose class is a night class.
# """
night_hours = {c: gp.quicksum(shift_hours[sh] * x[c, w, d, sh] for w in cycleWeeks for d in Weekdays for sh in shift_hours.keys() if sh in NIGHT_IDS) for c in cycles}

# difference variables:
# diff_night_pos + diff_night_neg = abs(night_hours[c] - night_hours[c+1]) for all c
diff_night_pos = modelCycle.addVars(pair_cycles, lb=0, name="diff_night_pos")
diff_night_neg = modelCycle.addVars(pair_cycles, lb=0, name="diff_night_neg")

# Compare each cycle with its direct neighbour (c vs c+1). Balancing every
# neighbour pair also balances the whole chain (c1 ~ c2 ~ c3 ...) by transitivity
for c in pair_cycles:
    # difference in night hours between cycle c and the next cycle
    diff = night_hours[c] - night_hours[c + 1]

    # Force (diff_night_pos - diff_night_neg) to equal that difference, so diff_night_pos + diff_night_neg becomes |difference|.
    # (active_cycle = 0), so we only balance cycles that actually exist.
    # [c37]
    modelCycle.addConstr(
        diff - (diff_night_pos[c] - diff_night_neg[c]) <=  huge_number * (1 - active_cycle[c + 1]),
        name=f"diff_night_upper_c{c}",
    )
    # [c38]
    modelCycle.addConstr(
        diff - (diff_night_pos[c] - diff_night_neg[c]) >= -huge_number * (1 - active_cycle[c + 1]),
        name=f"diff_night_lower_c{c}",
    )

# related objective term: sum of diffs (ie absolute difference) across all cycles and categories
obj_term_even_night_shift_distribution = gp.quicksum(diff_night_pos[c] + diff_night_neg[c] for c in pair_cycles)

#### objective: equal weekend-duty load across cycles

In [ ]:
# [010]
# ("Pesch6")
# objective: spread the weekend duty load evenly across all cycles
# "weekend duty" is any working shift SCHEDULED on a Saturday or 
# Sunday (calendar days 6 and 7 of the cycle week), regardless of the
# shift's name or class, measured in presence time (shift_hours). This uses the
# schedule position, so weekend clones and future shift types count automatically.
# note: by this approach shifts scheduled for Friday, but ending on 
# Saturday (night shifts), are not considered weekend duties

WEEKEND_DAYS = (6, 7)  # 6 = Saturday, 7 = Sunday (see DICT_WEEKDAYS)

weekend_hours = {c: gp.quicksum(shift_hours[sh] * x[c, w, d, sh] for w in cycleWeeks for d in WEEKEND_DAYS for sh in shift_hours.keys()) for c in cycles}

diff_weekend_pos = modelCycle.addVars(pair_cycles, lb=0, name="diff_weekend_pos")
diff_weekend_neg = modelCycle.addVars(pair_cycles, lb=0, name="diff_weekend_neg")

for c in pair_cycles:
    diff = weekend_hours[c] - weekend_hours[c + 1]
    # [c39]
    modelCycle.addConstr(
        diff - (diff_weekend_pos[c] - diff_weekend_neg[c]) <=  huge_number * (1 - active_cycle[c + 1]),
        name=f"diff_weekend_upper_c{c}",
    )
    # [c40]
    modelCycle.addConstr(
        diff - (diff_weekend_pos[c] - diff_weekend_neg[c]) >= -huge_number * (1 - active_cycle[c + 1]),
        name=f"diff_weekend_lower_c{c}",
    )

# related objective term
obj_term_even_weekend_duties = gp.quicksum(diff_weekend_pos[c] + diff_weekend_neg[c] for c in pair_cycles)

### objective function(s)

In [ ]:
"""
# NOTES:
# This part of the code sets up the objective function for the optimization model. 
# It defines a list of objective terms, each associated with a specific criterion 
# and its corresponding weight as defined by the user (via parameters.csv). 
# Depending on the value of the parameter "obj_by_priority", the model will either 
# use a lexicographic approach (one Gurobi objective per criterion) or a blended 
# approach (a single objective that is a weighted sum of all selected criteria). 
"""

OBJ_BY_PRIORITY = int(params["obj_by_priority"]) == 1

# scale objective terms with robust upper bounds so weighted sums are comparable
# across criteria with different units and growth orders.
Cmax = MAX_NB_CYCLES
Wmax = MAX_CYCLE_WEEKS
Hshift_max = max(shift_hours.values())
Hweek_max = 7 * Hshift_max
Pairs_all_max = Cmax * (Cmax - 1) // 2
Pairs_adj_max = max(0, Cmax - 1)

# upper-bound style scales (kept strictly > 0 to avoid division by zero)
S_nb_workers = max(1.0, float(Cmax * Wmax) / 2)
S_pesch0_avg_weekly_pres = max(1.0, float(Pairs_all_max * Hweek_max * 7))
S_pesch1_target_weekly_working_hrs = max(1.0, float(TARGET_WEEKLY_HOURS))/2
S_pesch2_equal_shift_cat_prop = max(1.0, float(2 * Cmax * Wmax * Hweek_max))
S_pesch3_change_of_types_cycle = max(1.0, float(Cmax * Wmax))
S_pesch4_equal_weekly_pres = max(1.0, float(Cmax * Hweek_max))
S_pesch5_changes_of_type_pan_cycles = max(1.0, float(Wmax * 7))
S_pesch6_even_weekend_duties = max(1.0, float(Pairs_adj_max * Wmax * 2 * Hshift_max))
S_pesch7_even_night_shifts = max(1.0, float(Pairs_adj_max * Wmax * Hweek_max))
S_pesch8_change_of_classes = max(1.0, float(Cmax * Wmax * 7))

"""
S_nb_workers = Cmax * Wmax
S_pesch0_avg_weekly_pres = p * (W * 7 * Hmax)
S_pesch1_target_weekly_working_hrs = 1
S_pesch2_equal_shift_cat_prop = max(1.0, float(2 * Cmax * Wmax * Hweek_max))
S_pesch3_change_of_types_cycle = 1
S_pesch4_equal_weekly_pres = max(1.0, float(Cmax * Hweek_max))
S_pesch5_changes_of_type_pan_cycles = 1
S_pesch6_even_weekend_duties = max(1.0, float(Pairs_adj_max * Wmax * 2 * Hshift_max))
S_pesch7_even_night_shifts = max(1.0, float(Pairs_adj_max * Wmax * Hweek_max))
S_pesch8_change_of_classes = 1
"""

objective_terms = [
    # (name, expression, weight, scale)
    ("nb_of_workers",               obj_term_min_nb_workers,                    W_Min_NB_WORKERS,                   S_nb_workers),
    ("avg_weekly_pres",             obj_term_equal_avg_weekly_presence_cycle,   W_EQUAL_AVG_WEEKLY_PRESENCE_CYCLE,  S_pesch0_avg_weekly_pres), # Pesch0
    ("target_weekly_working_hrs",   obj_term_target_weekly_hours,               W_TARGET_WEEKLY_HOURS,              S_pesch1_target_weekly_working_hrs), # Pesch1
    ("equal_shift_cat_prop",        obj_term_equal_shift_cat_proportion,        W_EQUAL_SHIFT_CAT_PROPORTION,       S_pesch2_equal_shift_cat_prop), # Pesch2
    ("change_of_types_cycle",       type_changes_total,                         W_MIN_CHANGEofTYPE_CYCLE,           S_pesch3_change_of_types_cycle), # Pesch3
    ("equal_weekly_pres",           obj_term_equal_weekly_presence_cycle,       W_EQUAL_WEEKLY_PRESENCE_CYCLE,      S_pesch4_equal_weekly_pres), # Pesch4
    ("changes_of_type_pan_cycles",  max_changes_pan_cycles * 1.0,               W_MIN_CHANGEofTYPE_CYCLEPLAN,       S_pesch5_changes_of_type_pan_cycles), # Pesch5
    ("even_weekend_duties",         obj_term_even_weekend_duties,               W_EVEN_WEEKEND_DUTIES,              S_pesch6_even_weekend_duties), # Pesch6
    ("even_night_shifts",           obj_term_even_night_shift_distribution,     W_EVEN_NIGHT_SHIFT_DISTRIBUTION,    S_pesch7_even_night_shifts), # Pesch7
    ("change_of_classes",           class_changes_total,                        W_MIN_CHANGEofCLASSES_d2d,          S_pesch8_change_of_classes), # Pesch8
]

# sort by 3rd tuple element (= weight) descending (for output)
objective_terms_sorted = sorted(objective_terms, key=lambda x: x[2], reverse=True)
objective_terms_active = [(name, expr, weight, scale) for name, expr, weight, scale in objective_terms_sorted if weight > 0.0]


if OBJ_BY_PRIORITY:
    # lexicographic: one Gurobi objective per criterion, 
    # priority = descending weight (higher weight = higher priority; ignored if 0!)
    # objectives with same weight are treated as one [blended] priority
    # normalization is applied here as well to avoid unintended magnitude bias
    for i, (name, expr, weight, scale) in enumerate(objective_terms_sorted):
        if weight > 0.0: # filter for objectives that have a positive weight (ie are actually selected by the user)
            modelCycle.setObjectiveN(expr / float(scale), index=i, priority=int(weight), weight=float(weight), name=name)
            print(f"priority: {int(weight)} - {name} / {int(scale):,}")
else:
    # blended: ONE plain objective = weighted sum of all criteria
    modelCycle.setObjective(gp.quicksum(float(weight) * expr/float(scale) for name, expr, weight, scale in objective_terms_active), GRB.MINIMIZE)
    print("blended objective: \n\t  " + "\n\t+ ".join([f"{weight:.2f} * {name} / {int(scale):,}" for name, expr, weight, scale in objective_terms_active]))


"""
if OBJ_BY_PRIORITY:
    # lexicographic: one Gurobi objective per criterion, 
    # priority = descending weight (higher weight = higher priority; ignored if 0!)
    # objectives with same weight are treated as one [blended] priority
    # normalization is applied here as well to avoid unintended magnitude bias
    for i, (name, expr, weight, scale) in enumerate(objective_terms_sorted):
        if weight > 0.0: # filter for objectives that have a positive weight (ie are actually selected by the user)
            modelCycle.setObjectiveN(expr, index=i, priority=int(weight), weight=float(weight), name=name)
            print(f"priority: {int(weight)} - {name}")
else:
    # blended: ONE plain objective = weighted sum of all criteria
    modelCycle.setObjective(gp.quicksum(float(weight) * expr for name, expr, weight, scale in objective_terms_active), GRB.MINIMIZE)
    print("blended objective: \n\t  " + "\n\t+ ".join([f"{weight:.2f} * {name}" for name, expr, weight, scale in objective_terms_active]))

"""
#print(f"objectives: {len(objective_terms_active)}, mode: {'priority (lexicographic)' if OBJ_BY_PRIORITY else 'weights (blended weighted sum)'}")


### solver configuration

In [ ]:
# solver settings (not part of the model, only how it is solved).
#   TimeLimit  -> stop and return the best solution so far
#   MIPFocus  = 1   -> prioritise finding good feasible solutions over proving the bound
#   Symmetry  = 2   -> aggressively detect and discard symmetric (identical) solutions
# The large MIP gap that remains is a weak lower bound, not a bad schedule.

# read solver limit settings from parameters.csv (user can set them to 0.0 to deactivate them)
solver_timeOut = float(params["solver_time_limit"]) # allows to deactivate the timeout by setting it to 0.0 in parameters.csv
solver_target_gap = float(params["solver_target_gap"]) # allows to deactivate the target gap by setting it to 0.0 in parameters.csv
if solver_timeOut > 0.0: # set this limit only if valid threshold was provided by user
    modelCycle.Params.TimeLimit = solver_timeOut
if solver_target_gap > 0.0: # set this limit only if valid threshold was provided by user
    modelCycle.Params.MIPGap = solver_target_gap
modelCycle.Params.MIPFocus = 2 # 1: prioritize (first) feasible solutions, 2: prioritize bound improvement, 3: balance both
modelCycle.Params.Heuristics = 0.5 # 0.0 = no heuristics, 1.0 = aggressive heuristics
modelCycle.Params.Cuts = 2 # 0 = no cuts, 1 = moderate cuts, 2 = aggressive cuts
modelCycle.Params.Symmetry = 2
modelCycle.ModelSense = GRB.MINIMIZE

### solver run

In [ ]:
#run optimizer

modelCycle.optimize()

### finish line

#### quick overview on results

In [ ]:
weeks_used = sum(active_cycle_week[c, s].X for c in cycles for s in cycleWeeks)
total_dev  = sum(dev_pos_time[c, w].X + dev_neg_time[c, w].X for c in cycles for w in cycleWeeks)

active_cnt   = sum(1 for c in cycles for w in cycleWeeks if active_cycle_week[c, w].X > 0.5)
on_target    = sum(1 for c in cycles for w in cycleWeeks
                   if active_cycle_week[c, w].X > 0.5 and dev_pos_time[c, w].X + dev_neg_time[c, w].X < 1e-6)
print(f"weeks on target (dev=0): {on_target} / {active_cnt} active weeks")
print(f"active weeks: {weeks_used:.0f}, total deviation (h): {total_dev:.1f}")

# Per-objective values, mode-independent: evaluate each criterion expression on the
# incumbent solution
for _name, _expr, _w, _ in objective_terms_sorted:
    print(f"objective {_name}: value {_expr.getValue():.1f} (weight {_w})")
print(f"\n")

# Diagnostic for scale quality: raw and normalized values on a common basis
print("normalized objective diagnostics (raw / scale -> normalized):")
for _name, _expr, _w, _scale in objective_terms_active:
    _raw = _expr.getValue()
    _norm = _raw / float(_scale)
    print(f"  {_name:26s} raw={_raw:10.2f}  scale={float(_scale):12.0f}  norm={_norm:10.10f}  w={_w}")
print(f"\n")

cat_imbalance = sum(dev_pos_cat[c, k].X + dev_neg_cat[c, k].X for c in cycles for k in categories)
print(f"Pesch2 share deviation, sum over cycles (h): {cat_imbalance:.1f}")
pesch8_changes = 0.5 * sum(class_diff[c, w, d, k].X for c in cycles for w in cycleWeeks for d in Weekdays for k in set_classes)
print(f"Pesch8 class changes: {pesch8_changes:.0f}")
pesch3_changes = 0.5 * sum(type_diff[c, w, d, t].X for c in cycles for w in cycleWeeks for d in Weekdays for t in set_types)
print(f"Pesch3 type changes (TA total): {pesch3_changes:.0f}")
pesch5_per_cycle = ", ".join(f"c{c}: {cycle_type_changes[c].getValue():.0f}" for c in cycles)
print(f"Pesch5 type changes per cycle (min-max target {max_changes_pan_cycles.X:.0f}): {pesch5_per_cycle}")
pesch0_spread = sum(dev_presence_pan_cycles[c1, c2].X for (c1, c2) in cycle_pairs)
print(f"Pesch0 presence imbalance between cycles (h): {pesch0_spread:.1f}")
pesch4_spread = ", ".join(f"c{c}: {max_presence_cycle[c].X - min_presence_cycle[c].X:.1f}" for c in cycles)
print(f"Pesch4 weekly presence spread inside cycles (h): {pesch4_spread}")
pesch7_imbalance = sum(diff_night_pos[c].X + diff_night_neg[c].X for c in pair_cycles)
print(f"Pesch7 night-load imbalance between cycles (h): {pesch7_imbalance:.1f}")


#### output to file

In [ ]:
# output
if modelCycle.SolCount > 0:   # if TimeLimit implemented -> status TIME_LIMIT, not OPTIMAL, so accept any found solution
    output_string = ""
    out_classes = sorted({dict_shift_to_class[sh] for sh in shift_hours})   # classes that carry hours
    with open(FOLDER_OUTPUT / "output_cycle.csv", "w") as file:             # "w" truncates previous runs
        file.write("created: " + str(dt.datetime.now()) + ";\n")
        file.write("FINAL CYCLE:\nMon;Tue;Wed;Thu;Fri;Sat;Sun;WorkHours;Deviation;OnTarget;PresenceHours;"
                   + ";".join(f"class_{k}_presence_h" for k in out_classes) + ";\n")
    for c in cycles:
        if active_cycle[c].X > 0.5:
            with open(FOLDER_OUTPUT / "output_cycle.csv", "a") as file:
                file.write(f"CYCLE: {c}:\n")
            for w in cycleWeeks:
                if active_cycle_week[c, w].X > 0.5:
                    for day in Weekdays:
                        for sh in Shifts:
                            if x[c, w, day, sh].X > 0.5:
                                output_string = f"{output_string}{sh.split('%_%', 1)[0]}({shift_info[sh]['start']}-{shift_info[sh]['end']});"
                    # per-week net work hours and deviation from the 40h target (from Pesch1 dev vars)
                    signed_dev = dev_pos_time[c, w].X - dev_neg_time[c, w].X
                    week_h     = TARGET_WEEKLY_HOURS + signed_dev
                    on_target  = "yes" if abs(signed_dev) < 1e-6 else "no"
                    output_string += f"{week_h:.1f};{signed_dev:+.1f};{on_target};"
                    # gross presence of the week: the reference total for the class columns
                    presence_h = sum(shift_hours[sh] * x[c, w, d, sh].X
                                     for d in Weekdays for sh in shift_hours)
                    output_string += f"{presence_h:.1f};"
                    # presence hours split by shift class (columns sum to PresenceHours)
                    for k in out_classes:
                        class_h = sum(shift_hours[sh] * x[c, w, d, sh].X
                                      for d in Weekdays for sh in shift_hours if dict_shift_to_class[sh] == k)
                        output_string += f"{class_h:.1f};"
                    with open(FOLDER_OUTPUT / "output_cycle.csv", "a") as file:
                        file.write(output_string + "\n")
                    output_string = ""
    
    output_string = ""
    if OBJ_BY_PRIORITY:
        for i, (name, expr, weight, scale) in enumerate(objective_terms_sorted):
            if weight > 0.0: 
                output_string += f"\npriority: {int(weight)} - {name}; value: {expr.getValue():.0f}"
        with open(FOLDER_OUTPUT / "output_cycle.csv", "a") as file:
            file.write("\n\nOBJECTIVES \n:" + output_string)

    else:
    # blended: ONE plain objective = weighted sum of all criteria
        output_string += ("\n\nblended objective: \n\t  " + "\n\t+ ".join([f"{weight:.2f} * {name}; value: {expr.getValue():.0f}" for name, expr, weight, scale in objective_terms_active]))
        with open(FOLDER_OUTPUT / "output_cycle.csv", "a") as file:
            file.write(output_string + "\n")